# CatBoost — Qualifying Position Predictor

CatBoost handles `power_unit` as a **native categorical** (no encoding needed) and `NaN` in numerical features natively.  
Hyperparameter tuning via **Optuna** (75 trials). Val years: 2021 / 2022 / 2023 | Test: 2024–2025.

In [ ]:
import pandas as pd
import numpy as np
import joblib, json, os, warnings
warnings.filterwarnings('ignore')

PROC_CSV = r'C:\Users\CLL\OneDrive\Documents\GitHub\F1-Race-Predictor\notebooks\f1_processed.csv'
MDL_DIR  = r'C:\Users\CLL\OneDrive\Documents\GitHub\F1-Race-Predictor\models'
os.makedirs(MDL_DIR, exist_ok=True)

# ── Feature columns (leakage-free — no Q1/Q2/Q3 gaps, no appearance flags) ──
FEATURE_COLS = [
    "reg_disruption_index", "years_since_reg_change",
    "season_stage_ratio", "driver_career_races", "driver_q3_rate_season",
    "driver_circuit_q_pos_hist", "prior_year_q_pos_same_circuit",
    "team_rolling_q_pos_5r", "driver_rolling_race_pos_5r", "teammate_q_gap_season",
    "is_night_race", "is_street_circuit",
    "driver_cum_pts", "team_cum_pts",
    "driver_pts_gap_to_leader", "team_pts_gap_to_leader",
    "driver_q_vs_race_delta_5r",
    "circuit_altitude_m", "circuit_length_km", "num_corners", "num_drs_zones",
    "driver_age", "is_home_race",
    "fp1_gap", "fp2_gap", "fp3_gap",
    "is_wet_qualifying", "track_temp_avg", "air_temp_avg", "humidity_avg", "wind_speed_avg",
    "has_fp_data",          # explicit NaN flag for 2023+ only features
]
TARGET = "GridPosition"
# Rolling-window validation years (train on all prior, validate on this year)
VAL_YEARS  = [2021, 2022, 2023]
TEST_YEARS = [2024, 2025]

## Load & Prepare Data

In [ ]:
df = pd.read_csv(PROC_CSV)

# Derived columns not saved to CSV
df["has_fp_data"] = df["fp1_gap"].notna().astype(int)

# Drop rows with missing target
df = df[df[TARGET].notna()].copy()
df = df.sort_values(["Season","Round"]).reset_index(drop=True)

print(f"Loaded: {len(df)} rows | seasons: {sorted(df['Season'].unique())}")
print(f"Missing values per feature:")
print(df[FEATURE_COLS].isna().sum()[df[FEATURE_COLS].isna().sum() > 0].to_string())

## Feature Preparation

`power_unit` kept as string — CatBoost handles it natively via `cat_features` index.

In [ ]:
from catboost import CatBoostRegressor, Pool

XCOLS = FEATURE_COLS + ["power_unit"]   # power_unit stays as string
CAT_IDX = [XCOLS.index("power_unit")]   # CatBoost needs index or name

X = df[XCOLS].copy()
# CatBoost requires NaN-filled strings for cat columns
X["power_unit"] = X["power_unit"].fillna("Unknown")
y = df[TARGET].values
seasons = df["Season"].values

## Hyperparameter Tuning (Optuna)

Search space: `iterations`, `depth`, `learning_rate`, `l2_leaf_reg`, `bagging_temperature`, `random_strength`, `border_count`. Early stopping (50 rounds) inside each fold.

In [ ]:
import optuna
from catboost import CatBoostRegressor, Pool
optuna.logging.set_verbosity(optuna.logging.WARNING)

def cb_objective(trial):
    params = dict(
        iterations         = trial.suggest_int("iterations", 300, 1500),
        depth              = trial.suggest_int("depth", 3, 8),
        learning_rate      = trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        l2_leaf_reg        = trial.suggest_float("l2_leaf_reg", 1e-8, 10.0, log=True),
        bagging_temperature= trial.suggest_float("bagging_temperature", 0.0, 5.0),
        random_strength    = trial.suggest_float("random_strength", 1e-8, 10.0, log=True),
        border_count       = trial.suggest_int("border_count", 32, 255),
        random_seed=42, verbose=0, loss_function="RMSE",
    )
    maes = []
    for val_yr in VAL_YEARS:
        tr = seasons < val_yr
        va = seasons == val_yr
        train_pool = Pool(X[tr], y[tr], cat_features=CAT_IDX)
        val_pool   = Pool(X[va], y[va], cat_features=CAT_IDX)
        m = CatBoostRegressor(**params)
        m.fit(train_pool, eval_set=val_pool,
              early_stopping_rounds=50, use_best_model=True)
        maes.append(np.mean(np.abs(m.predict(X[va]) - y[va])))
    return np.mean(maes)

study_cb = optuna.create_study(direction="minimize",
                                study_name="cb_quali",
                                sampler=optuna.samplers.TPESampler(seed=42))
study_cb.optimize(cb_objective, n_trials=75, show_progress_bar=True)

print(f"\nBest CV MAE: {study_cb.best_value:.4f}")
print("Best params:", study_cb.best_params)

## Final Model — Train & Evaluate

In [ ]:
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt

best_p = study_cb.best_params
best_p.update({"random_seed":42,"verbose":0,"loss_function":"RMSE"})

tr_mask = seasons <= 2023
te_mask = np.isin(seasons, TEST_YEARS)

train_pool = Pool(X[tr_mask], y[tr_mask], cat_features=CAT_IDX)
cb_final = CatBoostRegressor(**best_p)
cb_final.fit(train_pool)

preds_test = cb_final.predict(X[te_mask])
mae  = mean_absolute_error(y[te_mask], preds_test)
rmse = np.sqrt(np.mean((preds_test - y[te_mask])**2))
print(f"Test MAE : {mae:.4f}")
print(f"Test RMSE: {rmse:.4f}")

# Feature importance
fi = cb_final.get_feature_importance()
fig, axes = plt.subplots(1, 2, figsize=(15, 8))
order = np.argsort(fi)
axes[0].barh([XCOLS[i] for i in order], fi[order], color="#9b59b6")
axes[0].set_title("CatBoost feature importance (PredictionValuesChange)")
axes[0].set_xlabel("Importance")

axes[1].scatter(y[te_mask], preds_test - y[te_mask], alpha=0.4, s=12, color="#e74c3c")
axes[1].axhline(0, color="black", lw=0.8)
axes[1].set_xlabel("Actual GridPosition")
axes[1].set_ylabel("Residual (pred - actual)")
axes[1].set_title(f"Residuals on 2024-2025  (MAE={mae:.3f})")
plt.tight_layout(); plt.show()

fi_df = pd.DataFrame({"feature": XCOLS, "importance": fi}).sort_values("importance", ascending=False)
print("\nTop 10 features:")
print(fi_df.head(10).to_string(index=False))

## Save Model

In [ ]:
cb_final.save_model(os.path.join(MDL_DIR, "catboost_quali.cbm"))
with open(os.path.join(MDL_DIR, "catboost_best_params.json"), "w") as f:
    json.dump(study_cb.best_params, f, indent=2)
print("Saved: catboost_quali.cbm + catboost_best_params.json")